![Nuclio logo](https://cdn.prod.website-files.com/633b44aa19e0eb425d7e8c7f/63724d646ecb4bd4cbdf7440_LOGO%20NUCLIO%20DIGITAL%20SCHOOL-01-01-p-500.png)

## 1. Librerias & Google Drive

Para empezar carguemos esas librerias que nos hacen falta

In [ ]:
import tensorflow as tf
from tensorflow import keras as ks
from matplotlib import pyplot as plt
import numpy as np
import time
import datetime

Incorporamos la carpeta de Google Drive para poder "guardar" nuestros modelos

1. Abrir barra lateral de Carpeta
2.  Hacer click en carpeta con logo de Google Drive


In [ ]:
# Aquí iria el codigo para conectarse a Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 2. Arquitectura de red del modelo - Proyecto 0


In [ ]:
model = ks.Sequential()

model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), strides=1, activation='relu',
                           padding='same'))
model.add(ks.layers.MaxPooling2D((2, 2)))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(32, activation='relu'))
model.add(ks.layers.Dense(10, activation='softmax'))


Para revisar un modelo, nos basta con llamar al método **.summary()** del modelo

In [ ]:
model.summary()

## 3. Optimizador, función error


In [ ]:
model.compile(optimizer='Adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 4. Preparamos los datos

Cargamos los datos de CIFAR10 de lo que os paso por Google Drive

https://drive.google.com/file/d/18uT-D7Ak2bj_SZ0nKdXLYDYuKVaaFfzJ/view?usp=sharing

Subid el archivo `cifar-10-batches-py.tar.gz` a vuestro Google Drive, a una carpeta conocida que luego tendréis que adaptar.

Luego pasaremos a cargarlos y post-procesarlos para tener un dataset idéntico al que tendrías cargado directamente de Keras

Nota: cambiad el PATH del comando `! cp` que apunte donde esté el `cifar-10-batches-py.tar.gz` localizado

In [ ]:
! cp /content/drive/MyDrive/Entregable-4-Deep-Learning/cifar-10-batches-py.tar.gz /content/
! tar -xzf /content/cifar-10-batches-py.tar.gz -C /content/

! ls -la /content
! ls -la /content/cifar-10-batches-py


In [ ]:
def unpickle(file):
    import pickle
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

data_dir = "/content/cifar-10-batches-py/"
def load_data(data_dir):
#load training data
 for i in range(1, 6):
  filename = data_dir+"data_batch_"+str(i)
  dictionary = unpickle(filename)
  x_data = dictionary[b'data']
  y_data = np.array(dictionary[b"labels"])
  if i==1:
   x_train = x_data
   y_train= y_data
  else:
   x_train = np.concatenate((x_train, x_data), axis = 0)
   y_train = np.concatenate((y_train, y_data), axis = 0)
#load testing data
 filename = data_dir+"test_batch"
 dictionary = unpickle(filename)
 x_test = dictionary[b"data"]
 y_test = np.array(dictionary[b"labels"])
 return x_train, y_train, x_test, y_test

In [ ]:
def parse_record(record):
 depth_major = record.reshape((3, 32, 32))
 image = np.transpose(depth_major, [1, 2, 0])
 return image


In [ ]:
# Si se pudiera cargar directamente de Keras, serian estas dos única líneas, pero es muy lenta la descarga
# cifar10 = ks.datasets.cifar10
# x_train, y_train, x_test, y_test = cifar10.load_data(data_dir)

x_train_rows, y_train, x_test_rows, y_test = load_data(data_dir)


In [ ]:
# Validamos el resultado
print('Train: X=%s, y=%s' % (x_train_rows.shape, y_train.shape))
print('Test: X=%s, y=%s' % (x_test_rows.shape, y_test.shape))

In [ ]:
x_train =  []
x_test =  []

for i in range(x_train_rows.shape[0]):
  x_train.append(parse_record(x_train_rows[i]))

for i in range(x_test_rows.shape[0]):
  x_test.append(parse_record(x_test_rows[i]))

x_train = np.array(x_train)
x_test = np.array(x_test)

In [ ]:
# Validamos el resultado
print('Train: X=%s, y=%s' % (x_train.shape, y_train.shape))
print('Test: X=%s, y=%s' % (x_test.shape, y_test.shape))

In [ ]:
x_train_scaled, x_test_scaled = x_train / 255.0, x_test / 255.0

In [ ]:
# Validamos el resultado
print('Train: X=%s, y=%s' % (x_train_scaled.shape, y_train.shape))
print('Test: X=%s, y=%s' % (x_test_scaled.shape, y_test.shape))

Obtenemos un array con todas las labels de CIFAR10

In [ ]:
cifar10_labels = [
'airplane', # id 0
'automobile',
'bird',
'cat',
'deer',
'dog',
'frog',
'horse',
'ship',
'truck'
]

print('Number of labels: %s' % len(cifar10_labels))

Pintemos una muestra de las imagenes del dataset CIFAR10, a ver si se parece en algo a lo que esperamos.
Primero, vemos que tipos de datos tengo, después mapeamos esas matrices en una escala de grises utilizando el método **.get_cmap()** de PlotLy con los nueve primeros números del dataset.

In [ ]:
# Pintemos una muestra de las las imagenes del dataset MNIST

print('Train: X=%s, y=%s' % (x_train.shape, y_train.shape))
print('Test: X=%s, y=%s' % (x_test.shape, y_test.shape))

for i in range(9):

    plt.subplot(330 + 1 + i)
    plt.imshow(x_train[i], cmap=plt.get_cmap('gray'))
    plt.title(cifar10_labels[y_train[i]])

plt.subplots_adjust(hspace = 1)
plt.show()

Como vamos a querer ir haciendo validación a la vez que entrenamos (muy practico)

In [ ]:
x_val_scaled = x_train_scaled[-10000:]
y_val = y_train[-10000:]

x_train_scaled = x_train_scaled[:-10000]
y_train = y_train[:-10000]


Cuando añadimos convoluciones, necesitamos "pre-tratar los datos", porque **la convolución espera una matriz de 4 campos** (más parecido a "imagenes"), en el caso de MNIST. CIFAR10 ya tiene el shape adecuado, así que no tenemos que hacer nada, pero como siempre decimos: es mejor que comprobeis su tamaño.

Por eso, al salir de la Convolution, hay que hacer un Flatten, porque las capas FullDense esperan arrays, no matrices!!

Luego lo imprimimos para ver que todo está correcto

In [ ]:
# Validamos el resultado
print('Train: X=%s, y=%s' % (x_train_scaled.shape, y_train.shape))
print('Test: X=%s, y=%s' % (x_test_scaled.shape, y_test.shape))
print('Validation: X=%s, y=%s' % (x_val_scaled.shape, y_val.shape))

## 5. Entrenamiento

Ya podemos ponernos a entrenar el modelo!! Ojo, que hemos de entrenar contra los datos convertidos al formato que espera la Convolution.

Empezaremos con 30 epocs, es decir, 30 pasadas completas del dataset (que a su vez sera con mini-batches internamente), fijando un batch de 64

El metodo .fit() nos permite, definir además si disponemos de varias CPUs, GPUs, y si queremos ir validando datos a cada fin de epoch.

In [ ]:
t = time.perf_counter()

In [ ]:
history = model.fit(x_train_scaled, y_train, epochs=20,
                 batch_size= 512, validation_data=(x_val_scaled, y_val))

In [ ]:
elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))

print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados

Obtengamos una grafica de como el error y la accuracy van evolucionando en cada epoch en los datos de entrenamiento y en la validación

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

In [ ]:
# Añadido (no forma parte del Proyecto 0): guardo sus resultados para la comparativa final.
import json
ruta_resultados = '/content/drive/MyDrive/Entregable-4-Deep-Learning/resultados_iteracion.json'
resultados_test = {'0': acc}
historias = {'0': history.history}
parametros = {'0': model.count_params()}
with open(ruta_resultados, 'w') as f:
    json.dump({'test': resultados_test, 'historias': historias, 'parametros': parametros}, f)
print('Mejor epoch (menor val_loss): %d de %d' % (int(np.argmin(history.history['val_loss'])) + 1, len(history.history['loss'])))

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='val')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='val')
plt.legend()
plt.show()

El coste podemos ver que es estable y es en el epoch 15 que se fija en un valor. Además la accuracy baila alrededor de 0.10. No se puede decir que sea un modelo muy bueno.

Veamos que tipo de predicciones estoy obteniendo sobre el conjunto de test (vamos a pintar las imagenes y sus clasificaciones)

In [ ]:
predictions = model.predict(x_test_scaled)

Una de las ventajas de Python es que hay montones de funciones y código realizado por terceras personas. Aquí me he fusilado unas bonitas funciones (que he adaptado un poco a mis necesidades) para poder pintar las imagenes, su label (ground truth) y las clasificaciones que hemos realizado.

In [ ]:
def plot_image(i, predictions_array, true_label, img):
  predictions_array, true_label, img = predictions_array, true_label[i], img[i]
  plt.grid(False)
  plt.xticks([])
  plt.yticks([])

  plt.imshow(img, cmap=plt.cm.binary)

  predicted_label = np.argmax(predictions_array)
  if predicted_label == true_label:
    color = 'blue'
  else:
    color = 'red'

  plt.xlabel("{} {:2.0f}% ({})".format(predicted_label,
                                100*np.max(predictions_array),
                                true_label),
                                color=color)

def plot_value_array(i, predictions_array, true_label):
  predictions_array, true_label = predictions_array, true_label[i]
  plt.grid(False)
  plt.xticks(range(10))
  plt.yticks([])
  thisplot = plt.bar(range(10), predictions_array, color="#777777")
  plt.ylim([0, 1])
  predicted_label = np.argmax(predictions_array)

  thisplot[predicted_label].set_color('red')
  thisplot[true_label].set_color('blue')

Dibujamos las primeras imagenes, con las predicciones y sus valores reales (un total de 20 imagenes, para no abusar de vuestros laptops)

Coloreamos las prediciones correctas en azul y los fallos en rojo. Aunque primero he impreso las etiquetas para que tengamos una referencia al grafico.

In [ ]:
i = 0
for l in cifar10_labels:
    print(i, l)
    i += 1

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo para futuras evaluaciones

Id cambiando los nombres del archivo para ir guardando los diversos proyectos.
Aquí lo guardamos en "local", pero recordad que lo ideal es que lo guardeis en Google Drive (teneis que montar la unidad, fijar la ruta y allí guardar el modelo .h5)

In [ ]:
ruta_googledrive = "/content/drive/MyDrive/Entregable-4-Deep-Learning/" # A revisar en vuestra instancia

model.save(ruta_googledrive+'cifar10_p0.keras')

---
# Proyecto 1 — Primer bloque VGG

**Criterios comunes a partir de aquí** (no cuentan como proyecto): máximo de 100 epochs con Early Stopping (paciencia 15, nos quedamos con el mejor epoch) y semilla fija, para que cada modelo entrene hasta que deja de mejorar y todos partan del mismo punto.

**Cambio del proyecto:** la convolución suelta se sustituye por un bloque VGG (dos convoluciones y un MaxPooling), la capa densa pasa de 32 a 128 neuronas y los pesos se inicializan con he_uniform.

## 2. Arquitectura de red del modelo - Proyecto 1

In [ ]:
ks.backend.clear_session()      # empezamos de cero, sin restos del proyecto anterior
ks.utils.set_random_seed(42)   # mismo punto de partida para todos los proyectos

In [ ]:
model = ks.Sequential()

model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(128, activation='relu', kernel_initializer='he_uniform'))
model.add(ks.layers.Dense(10, activation='softmax'))

Para revisar un modelo, nos basta con llamar al método **.summary()** del modelo

In [ ]:
model.summary()

## 3. Optimizador, función error

In [ ]:
model.compile(optimizer='Adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 5. Entrenamiento

In [ ]:
# Early Stopping: si el loss de validación no mejora en 15 epochs seguidos, paramos.
# Con restore_best_weights=True el modelo se queda con los pesos del MEJOR epoch, no con los del último.
earlystopping = ks.callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                           restore_best_weights=True, verbose=1)

In [ ]:
t = time.perf_counter()

In [ ]:
history = model.fit(x_train_scaled, y_train, epochs=100,
                 batch_size= 512, validation_data=(x_val_scaled, y_val),
                 callbacks=[earlystopping])

In [ ]:
elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))

print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

In [ ]:
# Guardamos lo necesario para la comparativa final
mejor_epoch = int(np.argmin(history.history['val_loss'])) + 1
print('Mejor epoch: %d de %d' % (mejor_epoch, len(history.history['loss'])))

with open(ruta_resultados) as f:
    guardado = json.load(f)
guardado['test']['1'] = acc
guardado['historias']['1'] = {k: [float(x) for x in v] for k, v in history.history.items()}
guardado['parametros']['1'] = model.count_params()
with open(ruta_resultados, 'w') as f:
    json.dump(guardado, f)
print('Proyectos guardados:', list(guardado['test'].keys()))

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='val')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='val')
plt.legend()
plt.show()

In [ ]:
predictions = model.predict(x_test_scaled)

In [ ]:
i = 0
for l in cifar10_labels:
    print(i, l)
    i += 1

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo para futuras evaluaciones

In [ ]:
model.save(ruta_googledrive+'cifar10_p1.keras')

---
# Proyecto 2 — Segundo bloque VGG

## 2. Arquitectura de red del modelo - Proyecto 2

In [ ]:
ks.backend.clear_session()
ks.utils.set_random_seed(42)

model = ks.Sequential()
model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))

# Nuevo en este proyecto
model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(128, activation='relu', kernel_initializer='he_uniform'))
model.add(ks.layers.Dense(10, activation='softmax'))

model.summary()

## 3. Optimizador, función error - Proyecto 2

In [ ]:
model.compile(optimizer='Adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 5. Entrenamiento - Proyecto 2

In [ ]:
earlystopping = ks.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

t = time.perf_counter()

history = model.fit(x_train_scaled, y_train,
                    epochs=100,
                    batch_size=512,
                    validation_data=(x_val_scaled, y_val),
                    callbacks=[earlystopping])

elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))
print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados - Proyecto 2

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

mejor_epoch = int(np.argmin(history.history['val_loss'])) + 1
print('Mejor epoch: %d de %d' % (mejor_epoch, len(history.history['loss'])))

with open(ruta_resultados) as f:
    guardado = json.load(f)
guardado['test']['2'] = acc
guardado['historias']['2'] = {k: [float(x) for x in v] for k, v in history.history.items()}
guardado['parametros']['2'] = model.count_params()
with open(ruta_resultados, 'w') as f:
    json.dump(guardado, f)

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='validation')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='validation')
plt.legend()
plt.show()

In [ ]:
predictions = model.predict(x_test_scaled)

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test_scaled)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo - Proyecto 2

In [ ]:
model.save(ruta_googledrive + 'cifar10_p2.keras')

---
# Proyecto 3 — Dropout

## 2. Arquitectura de red del modelo - Proyecto 3

In [ ]:
ks.backend.clear_session()
ks.utils.set_random_seed(42)

model = ks.Sequential()
model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.2))

model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.3))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(128, activation='relu', kernel_initializer='he_uniform'))
model.add(ks.layers.Dropout(0.5))
model.add(ks.layers.Dense(10, activation='softmax'))

model.summary()

## 3. Optimizador, función error - Proyecto 3

In [ ]:
model.compile(optimizer='Adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 5. Entrenamiento - Proyecto 3

In [ ]:
earlystopping = ks.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

t = time.perf_counter()

history = model.fit(x_train_scaled, y_train,
                    epochs=100,
                    batch_size=512,
                    validation_data=(x_val_scaled, y_val),
                    callbacks=[earlystopping])

elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))
print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados - Proyecto 3

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

mejor_epoch = int(np.argmin(history.history['val_loss'])) + 1
print('Mejor epoch: %d de %d' % (mejor_epoch, len(history.history['loss'])))

with open(ruta_resultados) as f:
    guardado = json.load(f)
guardado['test']['3'] = acc
guardado['historias']['3'] = {k: [float(x) for x in v] for k, v in history.history.items()}
guardado['parametros']['3'] = model.count_params()
with open(ruta_resultados, 'w') as f:
    json.dump(guardado, f)

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='validation')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='validation')
plt.legend()
plt.show()

In [ ]:
predictions = model.predict(x_test_scaled)

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test_scaled)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo - Proyecto 3

In [ ]:
model.save(ruta_googledrive + 'cifar10_p3.keras')

---
# Proyecto 4 — Quitar el Dropout de las convoluciones

## 2. Arquitectura de red del modelo - Proyecto 4

In [ ]:
ks.backend.clear_session()
ks.utils.set_random_seed(42)

model = ks.Sequential()
model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))

model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(128, activation='relu', kernel_initializer='he_uniform'))
model.add(ks.layers.Dropout(0.5))
model.add(ks.layers.Dense(10, activation='softmax'))

model.summary()

## 3. Optimizador, función error - Proyecto 4

In [ ]:
model.compile(optimizer='Adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 5. Entrenamiento - Proyecto 4

In [ ]:
earlystopping = ks.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

t = time.perf_counter()

history = model.fit(x_train_scaled, y_train,
                    epochs=100,
                    batch_size=512,
                    validation_data=(x_val_scaled, y_val),
                    callbacks=[earlystopping])

elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))
print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados - Proyecto 4

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

mejor_epoch = int(np.argmin(history.history['val_loss'])) + 1
print('Mejor epoch: %d de %d' % (mejor_epoch, len(history.history['loss'])))

with open(ruta_resultados) as f:
    guardado = json.load(f)
guardado['test']['4'] = acc
guardado['historias']['4'] = {k: [float(x) for x in v] for k, v in history.history.items()}
guardado['parametros']['4'] = model.count_params()
with open(ruta_resultados, 'w') as f:
    json.dump(guardado, f)

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='validation')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='validation')
plt.legend()
plt.show()

In [ ]:
predictions = model.predict(x_test_scaled)

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test_scaled)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo - Proyecto 4

In [ ]:
model.save(ruta_googledrive + 'cifar10_p4.keras')

---
# Proyecto 5 — Tercer bloque VGG

## 2. Arquitectura de red del modelo - Proyecto 5

In [ ]:
ks.backend.clear_session()
ks.utils.set_random_seed(42)

model = ks.Sequential()
model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.2))

model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.3))

# Nuevo en este proyecto
model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.4))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(128, activation='relu', kernel_initializer='he_uniform'))
model.add(ks.layers.Dropout(0.5))
model.add(ks.layers.Dense(10, activation='softmax'))

model.summary()

## 3. Optimizador, función error - Proyecto 5

In [ ]:
model.compile(optimizer='Adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 5. Entrenamiento - Proyecto 5

In [ ]:
earlystopping = ks.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

t = time.perf_counter()

history = model.fit(x_train_scaled, y_train,
                    epochs=100,
                    batch_size=512,
                    validation_data=(x_val_scaled, y_val),
                    callbacks=[earlystopping])

elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))
print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados - Proyecto 5

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

mejor_epoch = int(np.argmin(history.history['val_loss'])) + 1
print('Mejor epoch: %d de %d' % (mejor_epoch, len(history.history['loss'])))

with open(ruta_resultados) as f:
    guardado = json.load(f)
guardado['test']['5'] = acc
guardado['historias']['5'] = {k: [float(x) for x in v] for k, v in history.history.items()}
guardado['parametros']['5'] = model.count_params()
with open(ruta_resultados, 'w') as f:
    json.dump(guardado, f)

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='validation')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='validation')
plt.legend()
plt.show()

In [ ]:
predictions = model.predict(x_test_scaled)

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test_scaled)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo - Proyecto 5

In [ ]:
model.save(ruta_googledrive + 'cifar10_p5.keras')

---
# Proyecto 6 — Learning rate más bajo (0,0005)

*Con un máximo de 100 epochs el entrenamiento se quedaba corto (mejor epoch 95 de 100), así que se re-entrenó con un máximo de 200 epochs. Según el enunciado, no cuenta como proyecto aparte.*

## 2. Arquitectura de red del modelo - Proyecto 6

In [ ]:
ks.backend.clear_session()
ks.utils.set_random_seed(42)

model = ks.Sequential()
model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.2))

model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.3))

model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.4))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(128, activation='relu', kernel_initializer='he_uniform'))
model.add(ks.layers.Dropout(0.5))
model.add(ks.layers.Dense(10, activation='softmax'))

model.summary()

## 3. Optimizador, función error - Proyecto 6

In [ ]:
from tensorflow.keras.optimizers import Adam

nuevo_adam = Adam(learning_rate=0.0005)   # antes 0.001, el valor por defecto de Adam

model.compile(optimizer=nuevo_adam,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 5. Entrenamiento - Proyecto 6

In [ ]:
earlystopping = ks.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

t = time.perf_counter()

history = model.fit(x_train_scaled, y_train,
                    epochs=200,
                    batch_size=512,
                    validation_data=(x_val_scaled, y_val),
                    callbacks=[earlystopping])

elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))
print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados - Proyecto 6

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

mejor_epoch = int(np.argmin(history.history['val_loss'])) + 1
print('Mejor epoch: %d de %d' % (mejor_epoch, len(history.history['loss'])))

with open(ruta_resultados) as f:
    guardado = json.load(f)
guardado['test']['6'] = acc
guardado['historias']['6'] = {k: [float(x) for x in v] for k, v in history.history.items()}
guardado['parametros']['6'] = model.count_params()
with open(ruta_resultados, 'w') as f:
    json.dump(guardado, f)

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='validation')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='validation')
plt.legend()
plt.show()

In [ ]:
predictions = model.predict(x_test_scaled)

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test_scaled)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo - Proyecto 6

In [ ]:
model.save(ruta_googledrive + 'cifar10_p6.keras')

---
# Proyecto 7 — Data augmentation

## 2. Arquitectura de red del modelo - Proyecto 7

In [ ]:
ks.backend.clear_session()
ks.utils.set_random_seed(42)

model = ks.Sequential()
model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.2))

model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.3))

model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.4))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(128, activation='relu', kernel_initializer='he_uniform'))
model.add(ks.layers.Dropout(0.5))
model.add(ks.layers.Dense(10, activation='softmax'))

model.summary()

## 3. Optimizador, función error - Proyecto 7

In [ ]:
from tensorflow.keras.optimizers import Adam

nuevo_adam = Adam(learning_rate=0.0005)

model.compile(optimizer=nuevo_adam,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 4. Data augmentation - Proyecto 7

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    horizontal_flip=True,       # volteo izquierda-derecha
    width_shift_range=0.1,      # desplazamiento horizontal de hasta un 10%
    height_shift_range=0.1      # desplazamiento vertical de hasta un 10%
)

# Los datos ya están divididos entre 255, así que no hace falta rescale
train_generator = train_datagen.flow(x_train_scaled, y_train, batch_size=512)

## 5. Entrenamiento - Proyecto 7

In [ ]:
earlystopping = ks.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

t = time.perf_counter()

history = model.fit(train_generator,
                    epochs=200,
                    validation_data=(x_val_scaled, y_val),
                    callbacks=[earlystopping])

elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))
print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados - Proyecto 7

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

mejor_epoch = int(np.argmin(history.history['val_loss'])) + 1
print('Mejor epoch: %d de %d' % (mejor_epoch, len(history.history['loss'])))

with open(ruta_resultados) as f:
    guardado = json.load(f)
guardado['test']['7'] = acc
guardado['historias']['7'] = {k: [float(x) for x in v] for k, v in history.history.items()}
guardado['parametros']['7'] = model.count_params()
with open(ruta_resultados, 'w') as f:
    json.dump(guardado, f)

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='validation')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='validation')
plt.legend()
plt.show()

In [ ]:
predictions = model.predict(x_test_scaled)

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test_scaled)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo - Proyecto 7

In [ ]:
model.save(ruta_googledrive + 'cifar10_p7.keras')

---
# Proyecto 8 — Batch size de 256

## 2. Arquitectura de red del modelo - Proyecto 8

In [ ]:
ks.backend.clear_session()
ks.utils.set_random_seed(42)

model = ks.Sequential()
model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.2))

model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.3))

model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.4))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(128, activation='relu', kernel_initializer='he_uniform'))
model.add(ks.layers.Dropout(0.5))
model.add(ks.layers.Dense(10, activation='softmax'))

model.summary()

## 3. Optimizador, función error - Proyecto 8

In [ ]:
from tensorflow.keras.optimizers import Adam

nuevo_adam = Adam(learning_rate=0.0005)

model.compile(optimizer=nuevo_adam,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 4. Data augmentation - Proyecto 8

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    horizontal_flip=True,       # volteo izquierda-derecha
    width_shift_range=0.1,      # desplazamiento horizontal de hasta un 10%
    height_shift_range=0.1      # desplazamiento vertical de hasta un 10%
)

# Los datos ya están divididos entre 255, así que no hace falta rescale
train_generator = train_datagen.flow(x_train_scaled, y_train, batch_size=256)   # antes 512

## 5. Entrenamiento - Proyecto 8

In [ ]:
earlystopping = ks.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

t = time.perf_counter()

history = model.fit(train_generator,
                    epochs=200,
                    validation_data=(x_val_scaled, y_val),
                    callbacks=[earlystopping])

elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))
print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados - Proyecto 8

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

mejor_epoch = int(np.argmin(history.history['val_loss'])) + 1
print('Mejor epoch: %d de %d' % (mejor_epoch, len(history.history['loss'])))

with open(ruta_resultados) as f:
    guardado = json.load(f)
guardado['test']['8'] = acc
guardado['historias']['8'] = {k: [float(x) for x in v] for k, v in history.history.items()}
guardado['parametros']['8'] = model.count_params()
with open(ruta_resultados, 'w') as f:
    json.dump(guardado, f)

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='validation')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='validation')
plt.legend()
plt.show()

In [ ]:
predictions = model.predict(x_test_scaled)

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test_scaled)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo - Proyecto 8

In [ ]:
model.save(ruta_googledrive + 'cifar10_p8.keras')

---
# Proyecto 9 — Capa densa de 512 neuronas

## 2. Arquitectura de red del modelo - Proyecto 9

In [ ]:
ks.backend.clear_session()
ks.utils.set_random_seed(42)

model = ks.Sequential()
model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.2))

model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.3))

model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.4))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(512, activation='relu', kernel_initializer='he_uniform'))   # antes 128
model.add(ks.layers.Dropout(0.5))
model.add(ks.layers.Dense(10, activation='softmax'))

model.summary()

## 3. Optimizador, función error - Proyecto 9

In [ ]:
from tensorflow.keras.optimizers import Adam

nuevo_adam = Adam(learning_rate=0.0005)

model.compile(optimizer=nuevo_adam,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 4. Data augmentation - Proyecto 9

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    horizontal_flip=True,       # volteo izquierda-derecha
    width_shift_range=0.1,      # desplazamiento horizontal de hasta un 10%
    height_shift_range=0.1      # desplazamiento vertical de hasta un 10%
)

# Los datos ya están divididos entre 255, así que no hace falta rescale
train_generator = train_datagen.flow(x_train_scaled, y_train, batch_size=512)

## 5. Entrenamiento - Proyecto 9

In [ ]:
earlystopping = ks.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

t = time.perf_counter()

history = model.fit(train_generator,
                    epochs=200,
                    validation_data=(x_val_scaled, y_val),
                    callbacks=[earlystopping])

elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))
print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados - Proyecto 9

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

mejor_epoch = int(np.argmin(history.history['val_loss'])) + 1
print('Mejor epoch: %d de %d' % (mejor_epoch, len(history.history['loss'])))

with open(ruta_resultados) as f:
    guardado = json.load(f)
guardado['test']['9'] = acc
guardado['historias']['9'] = {k: [float(x) for x in v] for k, v in history.history.items()}
guardado['parametros']['9'] = model.count_params()
with open(ruta_resultados, 'w') as f:
    json.dump(guardado, f)

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='validation')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='validation')
plt.legend()
plt.show()

In [ ]:
predictions = model.predict(x_test_scaled)

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test_scaled)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo - Proyecto 9

In [ ]:
model.save(ruta_googledrive + 'cifar10_p9.keras')

---
# Proyecto 10 — Learning rate por defecto (0,001)

## 2. Arquitectura de red del modelo - Proyecto 10

In [ ]:
ks.backend.clear_session()
ks.utils.set_random_seed(42)

model = ks.Sequential()
model.add(ks.layers.Input((32,32,3)))

model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.2))

model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(64, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.3))

model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.Conv2D(128, (3, 3), activation='relu', padding='same',
                           kernel_initializer='he_uniform'))
model.add(ks.layers.MaxPooling2D((2, 2)))
model.add(ks.layers.Dropout(0.4))

model.add(ks.layers.Flatten())
model.add(ks.layers.Dense(512, activation='relu', kernel_initializer='he_uniform'))
model.add(ks.layers.Dropout(0.5))
model.add(ks.layers.Dense(10, activation='softmax'))

model.summary()

## 3. Optimizador, función error - Proyecto 10

In [ ]:
from tensorflow.keras.optimizers import Adam

nuevo_adam = Adam(learning_rate=0.001)   # antes 0.0005; 0.001 es el valor por defecto de Adam

model.compile(optimizer=nuevo_adam,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## 4. Data augmentation - Proyecto 10

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    horizontal_flip=True,       # volteo izquierda-derecha
    width_shift_range=0.1,      # desplazamiento horizontal de hasta un 10%
    height_shift_range=0.1      # desplazamiento vertical de hasta un 10%
)

# Los datos ya están divididos entre 255, así que no hace falta rescale
train_generator = train_datagen.flow(x_train_scaled, y_train, batch_size=512)

## 5. Entrenamiento - Proyecto 10

In [ ]:
earlystopping = ks.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

t = time.perf_counter()

history = model.fit(train_generator,
                    epochs=200,
                    validation_data=(x_val_scaled, y_val),
                    callbacks=[earlystopping])

elapsed_time = datetime.timedelta(seconds=(time.perf_counter() - t))
print('Tiempo de entrenamiento:', elapsed_time)

## 6. Evaluamos los resultados - Proyecto 10

In [ ]:
_, acc = model.evaluate(x_test_scaled, y_test, verbose=0)
print('> %.3f' % (acc * 100.0))

mejor_epoch = int(np.argmin(history.history['val_loss'])) + 1
print('Mejor epoch: %d de %d' % (mejor_epoch, len(history.history['loss'])))

with open(ruta_resultados) as f:
    guardado = json.load(f)
guardado['test']['10'] = acc
guardado['historias']['10'] = {k: [float(x) for x in v] for k, v in history.history.items()}
guardado['parametros']['10'] = model.count_params()
with open(ruta_resultados, 'w') as f:
    json.dump(guardado, f)

In [ ]:
plt.title('Cross Entropy Loss')
plt.plot(history.history['loss'], color='blue', label='train')
plt.plot(history.history['val_loss'], color='orange', label='validation')
plt.legend()
plt.show()

plt.title('Classification Accuracy')
plt.plot(history.history['accuracy'], color='blue', label='train')
plt.plot(history.history['val_accuracy'], color='orange', label='validation')
plt.legend()
plt.show()

In [ ]:
predictions = model.predict(x_test_scaled)

num_rows = 5
num_cols = 4
start = 650
num_images = num_rows*num_cols
plt.figure(figsize=(2*2*num_cols, 2*num_rows))
for i in range(num_images):
  plt.subplot(num_rows, 2*num_cols, 2*i+1)
  plot_image(i+start, predictions[i+start], y_test, x_test_scaled)
  plt.subplot(num_rows, 2*num_cols, 2*i+2)
  plot_value_array(i+start, predictions[i+start], y_test)
plt.tight_layout()
plt.show()

## 7. Guardamos el modelo - Proyecto 10

In [ ]:
model.save(ruta_googledrive + 'cifar10_p10.keras')

# Acierto por clase sobre test (para el análisis final de la memoria)
clases = ['avión', 'automóvil', 'pájaro', 'gato', 'ciervo', 'perro', 'rana', 'caballo', 'barco', 'camión']
pred_clase = np.argmax(predictions, axis=1)
for c in range(10):
    m = (y_test == c)
    print('%-10s %.1f %%' % (clases[c], 100 * np.mean(pred_clase[m] == c)))